In [1]:
from lake_ice_helpers import open_era5_monthly_summary,lake_files
from count_analysis_helpers import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

* With this notebook you can create similar csv-dataset that was used as a base for final neural network training.
* Created and saved data frame named as 'df_full_clean' includes lake-data and era5 data from years 1960-2023.
* Each row in final data frame has lake-data and weather-summary from 2 years.
* We have weather statistics both from current year and previous year, when we consider 'year' as stated in our original data.
* Since we have 2 years for each observation row, we need to start at year 1961.
* Year 2024 does not include full weather observations, so we remove all data from this year.
* Lake-data is modified with function 'lake_files'. It includes all clean years withouth NaN-values when considering data from certain colums shown below.
* Target column will be ice_cover_duration, which is removed to be the last column of the 'df_full_clean'
* We implement neural network in different notebook.



In [ ]:
data_path = None #Add your path here
model_path = None #Add your path here
visualization_path = None #Add your path here

In [3]:
# Opening lake ice datafiles with funtion:
data_ip, ltbl_ice, ltbl_ice_depth, df_ice_clean = lake_files(data_path=data_path)
# Opening ERA5 summary table with function:
era5_summary_file = open_era5_monthly_summary(path_to_file=data_path+r'\era5_monthly_summary_celsius.csv')

In [4]:
df_ice_clean.head()

,station_id,year,ice_on,ice_off,ice_on_doy,ice_off_doy,ice_cover_duration,lake_id,country,lake_name,lat_wgs84,lon_wgs84,altitude_m,area_ha,depth_max_m,log_area_ha
0,4_1,1950,1949-12-10,1950-05-27,344.0,147.0,168.0,4,SE,Ajaure,65.510112,15.624015,439.0,1561.67,18.5,7.353511
1,4_1,1952,1951-11-16,1952-05-24,320.0,145.0,190.0,4,SE,Ajaure,65.510112,15.624015,439.0,1561.67,18.5,7.353511
2,4_1,1953,1952-10-23,1953-05-18,297.0,138.0,207.0,4,SE,Ajaure,65.510112,15.624015,439.0,1561.67,18.5,7.353511
3,4_1,1954,1953-11-22,1954-05-24,326.0,144.0,183.0,4,SE,Ajaure,65.510112,15.624015,439.0,1561.67,18.5,7.353511
4,4_1,1955,1954-11-10,1955-06-07,314.0,158.0,209.0,4,SE,Ajaure,65.510112,15.624015,439.0,1561.67,18.5,7.353511


In [5]:
df_ice_clean = pd.concat([df_ice_clean,
                         pd.get_dummies(df_ice_clean['country'],dtype=float)],axis=1)
df_ice_clean.head()

,station_id,year,ice_on,ice_off,ice_on_doy,ice_off_doy,ice_cover_duration,lake_id,country,lake_name,...,lon_wgs84,altitude_m,area_ha,depth_max_m,log_area_ha,CA,FI,NO,SE,US
0,4_1,1950,1949-12-10,1950-05-27,344.0,147.0,168.0,4,SE,Ajaure,...,15.624015,439.0,1561.67,18.5,7.353511,0.0,0.0,0.0,1.0,0.0
1,4_1,1952,1951-11-16,1952-05-24,320.0,145.0,190.0,4,SE,Ajaure,...,15.624015,439.0,1561.67,18.5,7.353511,0.0,0.0,0.0,1.0,0.0
2,4_1,1953,1952-10-23,1953-05-18,297.0,138.0,207.0,4,SE,Ajaure,...,15.624015,439.0,1561.67,18.5,7.353511,0.0,0.0,0.0,1.0,0.0
3,4_1,1954,1953-11-22,1954-05-24,326.0,144.0,183.0,4,SE,Ajaure,...,15.624015,439.0,1561.67,18.5,7.353511,0.0,0.0,0.0,1.0,0.0
4,4_1,1955,1954-11-10,1955-06-07,314.0,158.0,209.0,4,SE,Ajaure,...,15.624015,439.0,1561.67,18.5,7.353511,0.0,0.0,0.0,1.0,0.0


In [6]:
df_ice_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22279 entries, 0 to 22278
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   station_id          22279 non-null  object        
 1   year                22279 non-null  int64         
 2   ice_on              22279 non-null  datetime64[ns]
 3   ice_off             22279 non-null  datetime64[ns]
 4   ice_on_doy          22279 non-null  float64       
 5   ice_off_doy         22279 non-null  float64       
 6   ice_cover_duration  22279 non-null  float64       
 7   lake_id             22279 non-null  int64         
 8   country             22279 non-null  object        
 9   lake_name           22279 non-null  object        
 10  lat_wgs84           22279 non-null  float64       
 11  lon_wgs84           22279 non-null  float64       
 12  altitude_m          22279 non-null  float64       
 13  area_ha             22279 non-null  float64   

In [7]:
era5_summary_file.head()

tair_count_1  tair_count_2  tair_count_3  tair_count_4  \
station_id year                                                           
1000_1     1960          31.0          29.0          31.0          30.0   
           1961          31.0          28.0          31.0          30.0   
           1962          31.0          28.0          31.0          30.0   
           1963          31.0          28.0          31.0          30.0   
           1964          31.0          29.0          31.0          30.0   

                 tair_count_5  tair_count_6  tair_count_7  tair_count_8  \
station_id year                                                           
1000_1     1960          31.0          30.0          31.0          31.0   
           1961          31.0          30.0          31.0          31.0   
           1962          31.0          30.0          31.0          31.0   
           1963          31.0          30.0          31.0          31.0   
           1964          31.0          30.0          31.0          31.0   

                 tair_count_9  tair_count_10  ...  freeze_thaw_97.5%_3  \
station_id year                               ...                        
1000_1     1960          30.0           31.0  ...                  0.0   
           1961          30.0           31.0  ...                  1.0   
           1962          30.0           31.0  ...                  1.0   
           1963          30.0           31.0  ...                  1.0   
           1964          30.0           31.0  ...                  0.0   

                 freeze_thaw_97.5%_4  freeze_thaw_97.5%_5  \
station_id year                                             
1000_1     1960                  1.0                 0.00   
           1961                  1.0                 0.25   
           1962                  0.0                 0.00   
           1963                  1.0                 0.00   
           1964                  1.0                 0.00   

                 freeze_thaw_97.5%_6  freeze_thaw_97.5%_7  \
station_id year                                             
1000_1     1960                  0.0                  0.0   
           1961                  0.0                  0.0   
           1962                  0.0                  0.0   
           1963                  0.0                  0.0   
           1964                  0.0                  0.0   

                 freeze_thaw_97.5%_8  freeze_thaw_97.5%_9  \
station_id year                                             
1000_1     1960                  0.0                  0.0   
           1961                  0.0                  0.0   
           1962                  0.0                  0.0   
           1963                  0.0                  0.0   
           1964                  0.0                  0.0   

                 freeze_thaw_97.5%_10  freeze_thaw_97.5%_11  \
station_id year                                               
1000_1     1960                   1.0                 1.000   
           1961                   0.0                 1.000   
           1962                   1.0                 1.000   
           1963                   0.0                 1.000   
           1964                   0.0                 0.275   

                 freeze_thaw_97.5%_12  
station_id year                        
1000_1     1960                  0.25  
           1961                  0.25  
           1962                  0.00  
           1963                  0.25  
           1964                  0.00  

[5 rows x 480 columns]

In [8]:
era5_summary_file.loc[('1000_1',1960),:]

tair_count_1            31.00
tair_count_2            29.00
tair_count_3            31.00
tair_count_4            30.00
tair_count_5            31.00
                        ...  
freeze_thaw_97.5%_8      0.00
freeze_thaw_97.5%_9      0.00
freeze_thaw_97.5%_10     1.00
freeze_thaw_97.5%_11     1.00
freeze_thaw_97.5%_12     0.25
Name: (1000_1, 1960), Length: 480, dtype: float64

In [9]:
era5_summary_file.iloc[1,:]

tair_count_1            31.00
tair_count_2            28.00
tair_count_3            31.00
tair_count_4            30.00
tair_count_5            31.00
                        ...  
freeze_thaw_97.5%_8      0.00
freeze_thaw_97.5%_9      0.00
freeze_thaw_97.5%_10     0.00
freeze_thaw_97.5%_11     1.00
freeze_thaw_97.5%_12     0.25
Name: (1000_1, 1961), Length: 480, dtype: float64

In [10]:
era5_summary_file.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 77935 entries, ('1000_1', 1960) to ('9_1', 2024)
Columns: 480 entries, tair_count_1 to freeze_thaw_97.5%_12
dtypes: float64(480)
memory usage: 287.7+ MB


In [11]:
era5_summary_file.index.get_level_values(1).min() # Checking the minimum value for second element in MultiIndex tuples.

1960

In [12]:
def create_lake_era5_table(df_ice_clean,era5_summary_file,target_column):
    df_full_clean = pd.merge(df_ice_clean[df_ice_clean.year >= (era5_summary_file.index.get_level_values(1).min()+1)],
                            era5_summary_file,
                            how='left',
                            on=['station_id','year'])
    # Creating new MultiIndexing to add last years information to df_full_clean
    new_era5_index_tuples = [(first,second+1) for first,second in era5_summary_file.index]
    new_era5_multi_index = pd.MultiIndex.from_tuples(new_era5_index_tuples,names=['station_id','year'])
    df_full_clean = df_full_clean.dropna(axis=0)
    # Creating temporary era5 summary table with shifted indexes
    temporary_df = pd.DataFrame(era5_summary_file.to_numpy(),index=new_era5_multi_index,columns=era5_summary_file.columns)
    df_full_clean = pd.merge(df_full_clean,
                            temporary_df,
                            how='left',
                            on=['station_id','year'],
                            suffixes=['_current','_previous'])
    df_full_clean = df_full_clean.dropna(axis=0)
    first_columns = [x for x in df_full_clean.columns if x != target_column]
    df_full_clean = pd.concat([df_full_clean.loc[:,first_columns],df_full_clean.loc[:,target_column]],axis=1)
    return df_full_clean

In [13]:
df_full_clean = create_lake_era5_table(df_ice_clean=df_ice_clean,era5_summary_file=era5_summary_file,target_column='ice_cover_duration')

In [14]:
df_full_clean

,station_id,year,ice_on,ice_off,ice_on_doy,ice_off_doy,lake_id,country,lake_name,lat_wgs84,...,freeze_thaw_97.5%_4_previous,freeze_thaw_97.5%_5_previous,freeze_thaw_97.5%_6_previous,freeze_thaw_97.5%_7_previous,freeze_thaw_97.5%_8_previous,freeze_thaw_97.5%_9_previous,freeze_thaw_97.5%_10_previous,freeze_thaw_97.5%_11_previous,freeze_thaw_97.5%_12_previous,ice_cover_duration
0,4_1,1961,1960-11-01,1961-05-26,306.0,146.0,4,SE,Ajaure,65.510112,...,1.000,1.0,0.000,0.0,0.0,0.275,1.00,0.000,0.00,206.0
1,4_1,1962,1961-12-11,1962-05-23,345.0,143.0,4,SE,Ajaure,65.510112,...,1.000,1.0,0.000,0.0,0.0,0.000,1.00,1.000,0.25,163.0
2,4_1,1963,1962-11-18,1963-05-18,322.0,138.0,4,SE,Ajaure,65.510112,...,1.000,1.0,0.275,0.0,0.0,0.000,1.00,0.275,0.25,181.0
3,4_1,1964,1963-11-17,1964-05-18,321.0,139.0,4,SE,Ajaure,65.510112,...,1.000,0.0,0.000,0.0,0.0,0.000,1.00,0.000,1.00,183.0
4,4_1,1965,1964-11-26,1965-05-30,331.0,150.0,4,SE,Ajaure,65.510112,...,1.000,1.0,0.000,0.0,0.0,1.000,1.00,1.000,0.25,185.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18311,1196_1,2019,2018-12-07,2019-03-28,341.0,87.0,1196,US,Lake Wingra,43.053000,...,1.000,0.0,0.000,0.0,0.0,0.000,0.00,1.000,1.00,111.0
18312,1196_1,2020,2019-11-12,2020-03-12,316.0,72.0,1196,US,Lake Wingra,43.053000,...,0.000,0.0,0.000,0.0,0.0,0.000,0.25,1.000,1.00,121.0
18313,1196_1,2021,2020-12-06,2021-03-20,341.0,79.0,1196,US,Lake Wingra,43.053000,...,0.275,0.0,0.000,0.0,0.0,0.000,1.00,0.275,1.00,104.0
18314,1196_1,2022,2021-12-07,2022-03-25,341.0,84.0,1196,US,Lake Wingra,43.053000,...,0.275,0.0,0.000,0.0,0.0,0.000,0.00,1.000,1.00,108.0


In [15]:
df_full_clean.shape

(18316, 981)

In [16]:
df_full_clean.select_dtypes(include=['object']).columns

Index(['station_id', 'country', 'lake_name'], dtype='object')

In [17]:
df_ice_clean.columns

Index(['station_id', 'year', 'ice_on', 'ice_off', 'ice_on_doy', 'ice_off_doy',
       'ice_cover_duration', 'lake_id', 'country', 'lake_name', 'lat_wgs84',
       'lon_wgs84', 'altitude_m', 'area_ha', 'depth_max_m', 'log_area_ha',
       'CA', 'FI', 'NO', 'SE', 'US'],
      dtype='object')

In [18]:
drop_columns = ['station_id','ice_on','ice_off','ice_on_doy','ice_off_doy','lake_id','country']
keep_columns = [x for x in df_full_clean.columns if x not in drop_columns]

In [ ]:
sanity_checks = []
current_columns = [x for x in df_full_clean.columns if '_current' in x]
previous_columns = [x for x in df_full_clean.columns if '_previous' in x]

for i in range(len(df_full_clean)):
    row_original = df_full_clean.iloc[i,:]
    row_keep = row_original.loc[keep_columns]
    multi_index_current = (row_original['station_id'],row_original['year'])
    multi_index_previous = (row_original['station_id'],row_original['year']-1)
    current_check = np.allclose(era5_summary_file.loc[multi_index_current,:].values,row_keep.loc[current_columns].values.astype(float))
    previous_check = np.allclose(era5_summary_file.loc[multi_index_previous,:].values,row_keep.loc[previous_columns].values.astype(float))

    if current_check & previous_check:
        sanity_checks.append(True)
    else:
        sanity_checks.append(False)

In [20]:
np.sum(sanity_checks) == df_full_clean.shape[0]

True

In [21]:
df_full_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18316 entries, 0 to 18315
Columns: 981 entries, station_id to ice_cover_duration
dtypes: datetime64[ns](2), float64(974), int64(2), object(3)
memory usage: 137.1+ MB


Saving the produced data frame into csv-file:

In [ ]:
#df_full_clean.to_csv(data_path+'\\cleaned_lake_era5summary_for_network.csv')